# Get Sample -> Embed & cluster the admission medication column

This notebook embeds the admission medication text locally and clusters it with
KMeans, so you can draw a stratified sample for annotation.

**Pipeline:** `encode` (dense embeddings) -> KMeans -> stratified sample -> medoids

The medoid step comes **after** the sample, because the medoids serve as few-shot
examples for the `dynamic` prompting strategy and must therefore exclude every note
in the evaluation sample. Showing the model the annotated answer of a note it is
being evaluated on would leak the ground truth.

Everything runs on your machine. Your clinical text never leaves the computer.

---

### Before you start: download the model once (setup step, *outside* this notebook)

The only network access in this whole workflow is the one-time model download.
Do it once on a machine with internet, then this notebook runs fully offline:

```bash
uvx hf download sentence-transformers/all-MiniLM-L6-v2

```

This caches the weights (~80 MB) under `~/.cache/huggingface/`. If your analysis
environment is air-gapped, run the command on a connected machine and copy the
cache over. After that, the cells below never touch the network.

## 1. Import & Setup

**Force Offline Mode**

These env vars tell the Hugging Face libraries: *use only the local cache, never
the network.* If the model is cached, it loads normally; if it isn't, you get an
error instead of a silent download. That turns "nothing hits the network" into a
guarantee you can verify.

**This must be the first cell** - the libraries read these vars at import time, so
they have to be set before any huggingface import.

In [ ]:
import os

# Block both HF layers: the hub client and the transformers library.
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"

print("Offline mode set. The model must already be cached (see setup step above).")

In [ ]:
import json
from datetime import datetime, timezone

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from clinical_notes_extraction.config import PROJECT_ROOT

from sentence_transformers import SentenceTransformer

from sklearn.cluster import MiniBatchKMeans
from kneed import KneeLocator

## 2. Local Config

One place for the knobs you'll actually touch.

`random_state` is fixed everywhere KMeans is used: MiniBatchKMeans initialises its
centroids randomly and draws random mini-batches, so without a fixed seed the
cluster assignments - and therefore the sample drawn from them - would change on
every run. The same applies to the `.sample()` calls in the sampling step.

### 2.1. Local Variables

In [ ]:
DATASET_PATH = f'{PROJECT_ROOT}/data/datasets/structured_extraction/medication_on_admission_final_dataset.parquet'
TEXT_COLUMN  = "meds_on_admission_cleaned"   # the medication column to embed
ID_COLUMN    = "note_id"                     # used to keep embeddings aligned with rows

# Column carried into the medoid table as the few-shot example text.
# "text" is the full clinical note; switch to TEXT_COLUMN to carry only the
# medication section, which is what the extraction prompt actually asks about
# and costs far fewer tokens of the 8192-token context.
MEDOID_TEXT_COLUMN = "text"

In [ ]:
OUTPUT_DATA_PATH = f'{PROJECT_ROOT}/scripts/3_information_extraction/3_2_medications_on_admission/data'
CLUSTERING_PATH  = f'{OUTPUT_DATA_PATH}/clustering'

# Creates the entire folder structure; does nothing if they already exist
os.makedirs(OUTPUT_DATA_PATH, exist_ok=True)
os.makedirs(CLUSTERING_PATH, exist_ok=True)

### 2.2. Embed & Cluster Config

In [ ]:
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"       # a BERT model fine-tuned for sentence embeddings -> Sentence-BERT
RANDOM_STATE = 42
K_RANGE = range(2, 21)

# Embeddings are expensive on ~285K notes -> cache them.
# The ids file is what makes the cache safe to reuse (see 4.2).
EMB_CACHE     = f'{OUTPUT_DATA_PATH}/embeddings.npy'
EMB_IDS_CACHE = f'{OUTPUT_DATA_PATH}/embeddings_note_ids.npy'

# Clustering artefacts - needed to reproduce the partition and to reuse it downstream
# (dynamic prompting by medoids).
KMEANS_MODEL_PATH = f'{CLUSTERING_PATH}/kmeans_model.joblib'
CENTROIDS_PATH    = f'{CLUSTERING_PATH}/kmeans_centroids.npy'
INERTIAS_PATH     = f'{CLUSTERING_PATH}/elbow_inertias.csv'
METADATA_PATH     = f'{CLUSTERING_PATH}/clustering_metadata.json'

# Medoids: the shortlist goes out for manual annotation, the final table comes back
# with the annotation attached (section 6).
MEDOIDS_TO_ANNOTATE_PATH = f'{CLUSTERING_PATH}/medoids_to_annotate.parquet'
MEDOID_ANNOTATIONS_PATH  = f'{OUTPUT_DATA_PATH}/annotations/dynamic_prompts/medoids'
MEDOIDS_PATH             = f'{CLUSTERING_PATH}/medoids.parquet'

SAMPLE_PATH        = f'{OUTPUT_DATA_PATH}/sample.parquet'
FINAL_DATASET_PATH = f'{OUTPUT_DATA_PATH}/final_dataset.parquet'

In [ ]:
print(EMB_CACHE)
print(os.path.exists(EMB_CACHE))

## 3. Load Dataset

In [ ]:
df = pd.read_parquet(DATASET_PATH)
print(f"Loaded {df.shape[0]:,} rows x {df.shape[1]} columns")

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df1 = df.copy()

df1['meds_on_admission_cleaned_length'] = df1['meds_on_admission_cleaned'].str.len()

df1 = df1.drop(columns = ['meds_on_admission_length'])

df1.info()

The row order of `df1` is what the embedding matrix will be aligned to, so we fix it
explicitly by `note_id`. Without this, a parquet written by a different engine (or a
different upstream ordering) would silently permute the rows relative to a cached
embedding matrix.

In [ ]:
df1 = df1.sort_values(ID_COLUMN).reset_index(drop=True)

# note_id must be unique for the alignment guarantee below to mean anything
assert df1[ID_COLUMN].is_unique, f"{ID_COLUMN} is not unique - alignment cannot be guaranteed"

In [ ]:
df1['meds_on_admission_cleaned_length'].min()

In [ ]:
df1['meds_on_admission_cleaned_length'].max()

In [ ]:
df1[df1['meds_on_admission_cleaned_length']==5849]

### Empty / missing medication text

Notes with no medication text are embedded as the empty string, which produces one
single identical vector. Left unchecked they collapse into a degenerate cluster that
carries no semantic signal and distorts the elbow. We count them first and decide
explicitly rather than letting `fillna("")` hide the problem.

In [ ]:
n_missing = df1[TEXT_COLUMN].isna().sum()
n_blank   = (df1[TEXT_COLUMN].fillna("").str.strip() == "").sum()

print(f"Missing (NaN): {n_missing:,}")
print(f"Blank or whitespace-only: {n_blank:,}  ({n_blank / len(df1):.2%} of the corpus)")

In [ ]:
# Drop empty notes: they cannot be annotated for medication extraction anyway.
df1 = df1[df1[TEXT_COLUMN].fillna("").str.strip() != ""].reset_index(drop=True)
print(f"Notes retained: {len(df1):,}")

In [ ]:
texts    = df1[TEXT_COLUMN].astype(str).tolist()
note_ids = df1[ID_COLUMN].to_numpy()
print(f"Notes to embed: {len(texts):,}")

## 4. Embed

### 4.1. Load the model (offline)

If this cell errors, the model isn't cached yet - run the `hf download` setup step
first. If it loads cleanly with the offline vars set, that's your proof the
embedding step won't touch the network.

In [ ]:
print("Loading AI model")
print("=" * 50)

# Load model for embeddings
model = SentenceTransformer(EMBEDDING_MODEL_NAME)
print(f"Loaded {EMBEDDING_MODEL_NAME} | embedding dim = {model.get_embedding_dimension()}")

### 4.2. Encode (with a verified cache)

`normalize_embeddings=True` makes each vector unit-length, so KMeans' Euclidean
distance behaves like cosine similarity - the right metric for text embeddings.

We cache to disk so you never re-embed all ~285K notes between clustering experiments.

**The cache is only reused if it provably belongs to the current dataset.** We store
the `note_id` vector alongside the matrix and require an exact, order-sensitive match
before loading. A stale cache would not crash - it would silently align row *i* of the
current dataset with the embedding of a different note, corrupting every downstream
result. This check is what turns that silent failure into a loud one.

In [ ]:
def load_cached_embeddings(emb_path, ids_path, expected_ids):
    """Return the cached embedding matrix if it provably matches expected_ids, else None."""
    if not os.path.exists(emb_path):
        return None

    if not os.path.exists(ids_path):
        raise FileNotFoundError(
            f"Found {emb_path} but no companion id file at {ids_path}. "
            "This cache predates the alignment check and cannot be verified. "
            "Delete the .npy and re-embed."
        )

    cached_ids = np.load(ids_path, allow_pickle=True)
    if len(cached_ids) != len(expected_ids) or not np.array_equal(cached_ids, expected_ids):
        raise ValueError(
            f"Embedding cache is stale: it holds {len(cached_ids):,} notes and does not match "
            f"the current {len(expected_ids):,} notes (or their order). "
            f"Delete {emb_path} and {ids_path} to re-embed."
        )

    matrix = np.load(emb_path)
    if matrix.shape[0] != len(expected_ids):
        raise ValueError(
            f"Embedding matrix has {matrix.shape[0]:,} rows but {len(expected_ids):,} notes were expected."
        )
    return matrix

In [ ]:
emb = load_cached_embeddings(EMB_CACHE, EMB_IDS_CACHE, note_ids)

if emb is not None:
    print(f"Loaded cached embeddings from {EMB_CACHE} (alignment verified)")
else:
    emb = model.encode(
        texts,
        normalize_embeddings=True,   # L2 normalization done here
        batch_size=64,               # raise if you have GPU headroom
        show_progress_bar=True,
    ).astype(np.float32)
    np.save(EMB_CACHE, emb)
    np.save(EMB_IDS_CACHE, note_ids)
    print(f"Saved embeddings to {EMB_CACHE} and ids to {EMB_IDS_CACHE}")

print(f"Embedding matrix: {emb.shape}")   # (n_notes, 384) for MiniLM

`row_of` maps a `note_id` to its row in the embedding matrix. Every later step that
needs the vector of a specific note goes through this lookup rather than through
positional indexing, so a reordered or cached DataFrame cannot silently misalign.

In [ ]:
row_of = pd.Series(np.arange(len(note_ids)), index=note_ids)

### 4.3 Clustering

With embeddings computed for all notes, we now partition the population into clusters.

The goal is **not** to discover natural structure in the data - it is to create a discretized variable that will feed the composite stratification key used to draw the annotation sample.



#### 4.3.1 Choosing K with the elbow method

We sweep K over a range and record the **inertia** (within-cluster sum of squared distances to centroids) for each value. As K grows, inertia decreases monotonically - every additional cluster can only reduce (or leave unchanged) the total distance.

- The "elbow" is the point where the marginal gain from adding another cluster drops sharply: beyond it, we are paying in complexity without meaningful reduction in inertia.

- We use `MiniBatchKMeans` because the full `KMeans` would be prohibitively slow across a full sweep on ~285K notes.

- `n_init=10` runs each K with 10 different initializations and keeps the best, so the elbow curve is not contaminated by unlucky starts. `random_state=42` makes the sweep reproducible.

- The elbow is detected programmatically with `KneeLocator` (`curve="convex", direction="decreasing"`) rather than by visual inspection, to remove ambiguity.


In [ ]:
# Sweep K and record inertia (uses the in-memory matrix from 4.2 - already L2-normalized)
inertias = []

for k in K_RANGE:
    kmeans = MiniBatchKMeans(
        n_clusters=k,
        random_state=RANDOM_STATE,
        batch_size=4096,
        n_init=10,
    )
    kmeans.fit(emb)
    inertias.append(kmeans.inertia_)
    print(f"K={k:>2}: inertia={kmeans.inertia_:.0f}")

In [ ]:
# Persist the sweep: this table is the evidence behind the choice of K.
df_inertias = pd.DataFrame({"k": list(K_RANGE), "inertia": inertias})
df_inertias.to_csv(INERTIAS_PATH, index=False)
print(f"Saved elbow sweep to {INERTIAS_PATH}")
df_inertias

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(list(K_RANGE), inertias, marker="o")
ax.set_xlabel("K")
ax.set_ylabel("Inertia (within-cluster sum of squares)")
ax.set_title("Elbow method")
ax.set_xticks(list(K_RANGE))
ax.grid(alpha=0.3)
plt.show()

In [ ]:
# The inertia curve has no genuine elbow: it decreases almost linearly from K~5
# onward, and the non-monotonic steps at K=11 and K=17 show that MiniBatchKMeans
# sampling noise is of the same magnitude as the differences KneeLocator is
# reading. Automatic detection therefore returns 7 or 8 depending on minimal
# perturbations of the corpus. K is fixed manually at the value that produced the
# evaluation sample and the few-shot examples in use.
K_OVERRIDE = 8      # set to None to fall back on the detected elbow

kl = KneeLocator(
    list(K_RANGE),
    inertias,
    curve="convex",
    direction="decreasing",
)

print(f"KneeLocator suggests K={kl.elbow}")

if K_OVERRIDE is not None:
    K_CHOSEN = K_OVERRIDE
    print(f"Using K={K_CHOSEN} (manual override)")
    if kl.elbow is not None and kl.elbow != K_CHOSEN:
        print(f"  note: detected elbow ({kl.elbow}) differs from the fixed value")
else:
    if kl.elbow is None:
        raise ValueError(
            "KneeLocator found no elbow and no K_OVERRIDE was set. Inspect the plot "
            "above and fix K manually, documenting the reason."
        )
    K_CHOSEN = int(kl.elbow)
    if K_CHOSEN in (min(K_RANGE), max(K_RANGE)):
        print(f"WARNING: the elbow landed on the boundary of the sweep ({K_CHOSEN}).")

**Why K_RANGE = range(2, 21):**

- The lower bound of 2 is the smallest meaningful K - K=1 places every note in a single cluster, which gives you the total variance of the dataset as inertia and no partition to speak of. It is not informative for the elbow.

- The upper bound of 21 (so K goes up to 20) is a pragmatic ceiling.
    - Elbows in practice tend to appear at low K when they exist at all; extending the sweep to 30 or 50 mostly adds compute time and a long, flat tail that makes the elbow visually and algorithmically harder to locate - KneeLocator can get confused by very long post-elbow segments.

The cell above checks explicitly whether the elbow landed on either boundary of the sweep, which would be the signal to widen the range.

- A common convention in the literature is range(2, sqrt(n)) for small n, but with n=285K that heuristic is useless; a fixed practical ceiling around 20-30 is standard for datasets of this size.

#### 4.3.2. Fitting the final KMeans

Once K is fixed, we refit `MiniBatchKMeans` once with the same configuration and **attach the cluster labels to the notes DataFrame**.
- Labels are integers in `[0, K-1]` - the numbering is arbitrary (scikit-learn convention), not an ordering.


In [ ]:
kmeans = MiniBatchKMeans(
    n_clusters=K_CHOSEN,
    random_state=RANDOM_STATE,
    batch_size=4096,
    n_init=10,
)

cluster_labels = kmeans.fit_predict(emb)

df2 = df1.copy()
df2["cluster"] = cluster_labels

df2.head(10)

In [ ]:
np.sort(df2["cluster"].unique())

#### 4.3.3. Persisting the fitted model

The labels alone are not enough. Without the fitted estimator we cannot assign a
cluster to a note that was not in this run, and the centroids - which the medoid step
in section 6 needs - would disappear with the kernel.

We save three things:

- the **fitted estimator** (`joblib`), so any new note can be assigned with `.predict()`;
- the **centroids** as a plain `.npy`, so the vectors stay readable even if the
  scikit-learn version changes and unpickling the estimator breaks;
- a small **metadata** file recording the embedding model, K, the sweep range, the
  seed and the corpus size, so the partition can be traced back from the thesis.

In [ ]:
joblib.dump(kmeans, KMEANS_MODEL_PATH)
np.save(CENTROIDS_PATH, kmeans.cluster_centers_)

metadata = {
    "embedding_model": EMBEDDING_MODEL_NAME,
    "embedding_dim": int(emb.shape[1]),
    "n_notes": int(emb.shape[0]),
    "normalize_embeddings": True,
    "k_range": [min(K_RANGE), max(K_RANGE)],
    "k_chosen": K_CHOSEN,
    "elbow_method": "KneeLocator(curve='convex', direction='decreasing')",
    "algorithm": "MiniBatchKMeans",
    "batch_size": 4096,
    "n_init": 10,
    "random_state": RANDOM_STATE,
    "created_at": datetime.now(timezone.utc).isoformat(),
}

with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Saved model to      {KMEANS_MODEL_PATH}")
print(f"Saved centroids to  {CENTROIDS_PATH}  {kmeans.cluster_centers_.shape}")
print(f"Saved metadata to   {METADATA_PATH}")

#### 4.3.4 Distribution of notes across clusters

Before moving to stratified sampling, we inspect the distribution of notes across clusters.

A severely unbalanced partition (e.g. one cluster holding the majority of notes) is not a bug - KMeans is only a discretizer here.

In [ ]:
absolute_frequency = df2["cluster"].value_counts().sort_index()
relative_frequency = (df2["cluster"].value_counts(normalize=True).sort_index() * 100).round(2)

pd.DataFrame({
    "Absolute Frequency (number of notes per cluster)": absolute_frequency,
    "Relative Frequency (%)": relative_frequency,
})

## 5. Building the composite stratification key

The stratified sampling requires a discrete key that captures the axes along which we want to guarantee coverage of the annotation subset. We already have `cluster` from the KMeans step.

We now add a second axis: a proxy for note complexity based on the **length of the admission medication text**.

- We use text length rather than a direct medication count because the admission medication field is inconsistently formatted across notes (free text, line breaks, numbered lists), which makes any parsing-based count unreliable. Since parsing this field is precisely what the downstream extraction pipeline is meant to evaluate, computing a "true" count upstream would be circular. Text length is an imperfect but tractable substitute: longer blocks tend to correspond to heavier medication profiles, and the metric is uniform across formats.

- We discretize the length into four classifications using quartiles (`pd.qcut` with `q=4`), which guarantees balanced classifications by construction (~25% of the population in each). This is important for the composite key: unbalanced classifications would collapse strata once crossed with `cluster`, leaving combinations with too few notes to sample from.

- The labels `short`, `medium`, `long`, `very_long` produce an ordered categorical variable - the ordinal semantics are preserved in groupbys and value counts, while the string labels remain readable in inspection outputs and in the final thesis tables.

In [ ]:
df2.info()

In [ ]:
df2.info()

In [ ]:
df2["meds_on_admission_cleaned_length_classification"] = pd.qcut(
    df2["meds_on_admission_cleaned_length"],
    q=4,
    labels=["short", "medium", "long", "very_long"],
)

df2.info()

In [ ]:
sorted(df2['meds_on_admission_cleaned_length_classification'].unique())

**Stratified key:**
- cluster + meds_on_admission_cleaned_length_binary


### 5.2 Sampling design

The annotation budget is fixed at 32 notes due to constraints on manual annotation effort available within the thesis timeline. It imposes hard constraints on the stratification scheme.

Crossing `cluster` with the four length quartiles would produce more strata than the budget allows to populate with more than one note each, so we coarsen the length axis:

- **`cluster`** (K levels) - retained at full granularity, as it is the primary discretizer of the semantic space of medication text.
- **`med_length_binary`** (2 levels) - the four length quartiles are collapsed into `shorter` (short + medium) and `longer` (long + very_long). This preserves the length axis as a stratification signal while keeping the total number of strata tractable.

With K=8 this produces **8 x 2 = 16 strata** and allows **2 notes per stratum**, giving minimal within-stratum replication rather than the single-note-per-stratum regime that a finer key would force.

Before sampling, we enforce a **one-note-per-patient** constraint to eliminate the copy-forward leakage risk: patients with multiple admissions often have near-identical admission medication lists across notes, which would bias the sample toward duplicated content if left unfiltered. The constraint is applied *before* the stratified draw so that no patient can appear twice in the final sample.

#### Caveats on this sample size

Thirty-two notes constitute a **pilot annotation** rather than a statistically robust evaluation set.

Metrics computed on this sample will carry wide confidence intervals (bootstrap CIs of approximately +/-10-15 points on F1) and should be interpreted as directional signal, not point estimates. The primary value of this sample is qualitative: identifying failure modes of the extraction pipeline across the medication profile space, and providing evidence to justify further annotation effort if warranted.

In [ ]:
df2.info()

In [ ]:
NOTES_PER_STRATUM = 2

SAMPLE_COLUMNS = [
    'note_id',
    'subject_id',
    'text',
    'meds_on_admission_cleaned',
    'meds_on_admission_cleaned_length',
    'cluster',
    'meds_on_admission_cleaned_length_classification',
    'meds_on_admission_cleaned_length_binary',
]

STRATUM_KEYS = ["cluster", "meds_on_admission_cleaned_length_binary"]

In [ ]:
if not os.path.exists(FINAL_DATASET_PATH):
    df_final = df2.copy()

    df_final["meds_on_admission_cleaned_length_binary"] = df_final["meds_on_admission_cleaned_length_classification"].map({
        "short": "shorter", "medium": "shorter",
        "long": "longer", "very_long": "longer",
    })

    df_final.to_parquet(FINAL_DATASET_PATH)
    print('New Final Dataset recorded ... \n\n')


else:
    df_final = pd.read_parquet(FINAL_DATASET_PATH)
    print('Existing Final Dataset ... \n\n')


df_final.head(10)

#### 5.2.1 Stratum sizes before the draw

`groupby(...).sample(n=2)` raises if any stratum holds fewer than 2 rows, and the
one-note-per-patient filter runs *before* the draw, so a stratum that looks well
populated in the full corpus can still be thinned below the threshold. We therefore
materialise the deduplicated pool first and inspect the stratum sizes explicitly.

The table below is also the one to carry into the dissertation: it documents the
population each sampled pair was drawn from.

The rows are sorted by `note_id` before the per-patient draw. `DataFrame.sample` picks
rows by position, so a different upstream row order would yield a different sample from
the same seed; sorting on a stable key makes the draw reproducible.

In [ ]:
df_pool = (
    df_final[SAMPLE_COLUMNS]
    .sort_values("note_id")
    .groupby("subject_id", group_keys=False)
    .sample(n=1, random_state=RANDOM_STATE)
)

print(f"Pool after one-note-per-patient: {len(df_pool):,} notes "
      f"(from {len(df_final):,})")

In [ ]:
stratum_sizes = (
    df_pool.groupby(STRATUM_KEYS, observed=True)
    .size()
    .rename("n_available")
    .reset_index()
    .sort_values(STRATUM_KEYS)
)

expected_strata = df_pool["cluster"].nunique() * df_pool["meds_on_admission_cleaned_length_binary"].nunique()
print(f"Populated strata: {len(stratum_sizes)} of {expected_strata} possible")
print(f"Requested per stratum: {NOTES_PER_STRATUM} -> target sample size: "
      f"{len(stratum_sizes) * NOTES_PER_STRATUM}")

stratum_sizes

In [ ]:
undersized = stratum_sizes[stratum_sizes["n_available"] < NOTES_PER_STRATUM]

if len(undersized) > 0:
    print("The following strata cannot supply the requested number of notes:")
    print(undersized.to_string(index=False))
    raise ValueError(
        f"{len(undersized)} stratum/strata hold fewer than {NOTES_PER_STRATUM} notes. "
        "Lower NOTES_PER_STRATUM, merge the affected strata, or revisit the key."
    )

print(f"All {len(stratum_sizes)} strata can supply {NOTES_PER_STRATUM} notes.")

In [ ]:
print("K_CHOSEN:", K_CHOSEN)
print("clusters no pool:", sorted(df_pool["cluster"].unique()))
print("clusters no final_dataset:", sorted(df_final["cluster"].unique()))

In [ ]:
if not os.path.exists(SAMPLE_PATH):
    df_sample = (
        df_pool
        .groupby(STRATUM_KEYS, group_keys=False, observed=True)
        .sample(n=NOTES_PER_STRATUM, random_state=RANDOM_STATE)
    )

    # df_sample.to_parquet(SAMPLE_PATH)
    print('New Sample ... \n\n')

else:
    df_sample = pd.read_parquet(SAMPLE_PATH)
    print('Existing Sample ... \n\n')

df_sample.sort_values(by='meds_on_admission_cleaned_length', ascending=True)

In [ ]:
df_sample.info()

#### 5.2.2 Attaching the embedding to the sample

The `dynamic` strategy needs the vector of each evaluation note in order to rank the
few-shot candidates. Storing it on the sample means the extraction run never has to
load the embedding model at all - the comparison becomes one matrix product.

The join is by `note_id`, never by position: `df_sample` may have been read back from
parquet, in which case its index has no relationship to the rows of the embedding
matrix. The membership check that precedes the join catches a sample drawn before the
empty-note filter was introduced, whose ids may no longer exist in the corpus.

In [ ]:
missing_in_corpus = set(df_sample["note_id"]) - set(row_of.index)

if missing_in_corpus:
    raise ValueError(
        f"{len(missing_in_corpus)} sample note(s) are absent from the embedded corpus: "
        f"{sorted(missing_in_corpus)[:5]}. The sample predates the current dataset - "
        "either regenerate it or reconcile the ids before continuing."
    )

sample_rows = row_of.loc[df_sample["note_id"]].to_numpy()
df_sample["embedding"] = list(emb[sample_rows].astype(np.float32))

df_sample.head()


In [ ]:
df_sample.to_parquet(SAMPLE_PATH)
print(f"Embedding attached and sample saved to {SAMPLE_PATH}")

df_sample[["note_id", "cluster", "embedding"]].head()

A one-off check that the vector attached to a row really is that note's own
embedding, and not a neighbouring row's. Re-encodes a single note and compares.

In [ ]:
_probe_id = df_sample["note_id"].iloc[0]
_probe_text = df1.loc[df1[ID_COLUMN] == _probe_id, TEXT_COLUMN].iloc[0]
_probe_vec = model.encode([_probe_text], normalize_embeddings=True)[0]

print("Alignment check:", np.allclose(df_sample["embedding"].iloc[0], _probe_vec, atol=1e-5))

#### 5.2.3 Composition of the drawn sample

A final check that the draw did what the design says it does: one note per patient,
the requested number per stratum, and every stratum represented.

In [ ]:
print(f"Sample size: {len(df_sample)}")
print(f"Distinct patients: {df_sample['subject_id'].nunique()} (must equal the sample size)")
print(f"Distinct notes: {df_sample['note_id'].nunique()} (must equal the sample size)")

df_sample.groupby(STRATUM_KEYS, observed=True).size().rename("n_sampled").reset_index()